# **Atividade II - Análise Exploratória de Dados**

## **1 Importar bibliotecas e dados**

### 1.1 Importação de bibliotecas

In [ ]:
# Importando bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Bibliotecas importadas com sucesso!")

### 1.2 Importação de dados

In [ ]:
# Caminho do arquivo CSV
CAMINHO_ARQUIVO = r"..\data\469001_Chuvas.csv"
print("Caminho do arquivo CSV definido")

In [ ]:
# Criação do dataFrame inicial a partir do arquivo CSV
df = pd.read_csv(CAMINHO_ARQUIVO, skiprows=14, sep=';', decimal=',', encoding='latin1')
print("DataFrame inicial criado com sucesso!")

# Visualizando as primeiras linhas do DataFrame (função head())
print("\nExibindo as primeiras linhas do DataFrame inicial:")
df.head()

In [ ]:
# Cópia de segurança do DataFrame original
df_original = df.copy()
print("Cópia de segurança do DataFrame original criada com sucesso!")

## **2 Estrutura da Base de Dados**

### 2.1 Tamanho do DataFrame

In [ ]:
# Tamanho do DataFrame
print("Tamanho do DataFrame")
print(f"Qtd. linhas: {df.shape[0]} linhas")
print(f"Qtd. colunas: {df.shape[1]} colunas")

### 2.2 Propriedades das Colunas

In [ ]:
# Nome das colunas
print("Nome das Colunas:")
lista_colunas = df.columns.tolist()
print(lista_colunas)

In [ ]:
# Visualização dos tipos de dados
print("Tipos de Dados por Coluna:")
df.dtypes

### 2.3 Exemplos de registros

In [ ]:
# Exemplos de registros
print("Exibindo as primeiras linhas do DataFrame:")
df.head()

**Interpretação:**

**O que cada linha representa:** Cada linha é um **mês** de dados da estação pluviométrica 469001.

**Principais variáveis:**
- `NivelConsistencia`: qualidade dos dados (1 - Bruto, 2 - Consistido)
- `Data`: mês e ano da medição
- `Maxima`: maior chuva em um único dia no mês (mm)
- `Total`: total de chuva no mês (mm)
- `DiaMaxima`: o dia do mês em que ocorreu a maior precipitação
- `NumDiasDeChuva`: quantos dias choveu no mês
- `Chuva01` a `Chuva31`: chuva diária (mm)

## **3 Qualidade dos dados**

**Instrução:**\
Verificar valores ausentes, duplicatas, tipos incorretos, valores inconsistentes e valores extremos

### 3.1 Verificação de valores ausentes

In [ ]:
# Verificação de valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())

# Registro de colunas com valores ausentes
col_valores_ausentes = df.columns[df.isnull().any()].tolist()
print("\nResumo: Colunas - valores ausentes:")
print(f"Nome das colunas: {col_valores_ausentes}")
print(f"Total: {len(col_valores_ausentes)}")


### 3.2 Verificação de linhas duplicadas

In [ ]:
# Verificação de linhas duplicadas
print("\nVerificação de linhas duplicadas:")
num_linhas_duplicadas = df.duplicated().sum()
print(f"Número de linhas duplicadas: {num_linhas_duplicadas}")

# Flag de verificação de linhas duplicadas
linhas_duplicadas = False
if num_linhas_duplicadas > 0:
    linhas_duplicadas = True
print("\nFlag de verificação de linhas duplicadas criada com sucesso!")

### 3.3 Verificação de tipos incorretos

**Nota:** A verificação será efetuada apenas nas colunas de interesse

In [ ]:
# Definição das colunas de interesse
colunas_chuva = lista_colunas[13:44]
colunas_num_principais = ['Maxima', 'Total', 'NumDiasDeChuva']
colunas_interesse = ['Data'] + colunas_num_principais + colunas_chuva
#print(f"Colunas de chuva: {colunas_chuva}")
print(f"Colunas de interesse: {colunas_interesse}")

# Verificação de tipos incorretos
print("\nVerificação de tipos de dados incorretos")
for col in colunas_interesse:
    print(f"   - '{col}': {df[col].dtype} (Exemplo: {df[col].iloc[0]})")


### 3.4 Verificação de valores inconsistentes

**Nota:** A verificação será aplicada apenas a colunas categóricas de interesse

In [ ]:
# Definindo colunas de interesse
col_chuva_status = lista_colunas[44:76]
col_categoricas = ['NivelConsistencia', 'TipoMedicaoChuvas', 'MaximaStatus', 'TotalStatus'] + col_chuva_status
print("Colunas categoricas de interesse definidas")

# Metadados
dominios = {
    'NivelConsistencia': [1, 2],
    'TipoMedicaoChuvas': [1, 2, 3]
}
print("\nDicionário de metadados computado!")

In [ ]:
# Verificação de valores fora do padrão
contagens = {}

for col in col_categoricas:
    if col not in df.columns:
        continue

    permitidos = dominios.get(col, [0, 1, 2, 3, 4] if 'Status' in col else None)
    if permitidos is None:
        continue

    invalidos = (~df[col].isin(permitidos)).sum()

    if invalidos > 0:
        contagens[col] = invalidos
        print(f"{col}: {invalidos} valores fora do padrão.")

print("Validação concluída.")

### 3.5 Verificação de valores extremos

In [ ]:
# Valores extremos - estatística descritiva
print("\nEstatística descritiva p/ encontrar valores extremos:")
colunas_num_principais = ['Maxima', 'Total', 'NumDiasDeChuva']
print(df[colunas_num_principais].describe())

## **4 Tratamento dos Dados**

### 4.1 Remoção de Duplicatas

In [ ]:
# Verificação e Remoção de Duplicatas
if linhas_duplicadas:
    df = df.drop_duplicates()
    print("Linhas duplicadas removidas:")
    print(f"   - Qtd. de linhas: {df.shape[0]}")
else:
    print("Não havia linhas duplicadas")

### 4.2 Tratamento de Tipos incorretos

In [ ]:
# Converter a coluna 'Data' para o tipo datetime
df['Data'] = pd.to_datetime(df['Data'], format='%d/%m/%Y')
print("Coluna 'Data' convertida para o tipo datetime com sucesso")

### 4.3 Tratamento de Valores Ausentes

#### 4.3.1 Criação de DataFrame Temporário de Chuvas Diárias

In [ ]:
# Criação do dataFrame
df_chuvas = df[['EstacaoCodigo', 'NivelConsistencia', 'Data', 'Maxima', 'Total', 'NumDiasDeChuva'] + colunas_chuva].copy()
print("DataFrame Temporário de Chuvas criado com sucesso!")

# Visualização do dataFrame
df_chuvas.head()

#### 4.3.2 Tratamento de Valores Ausentes no DataFrame Temporário

In [ ]:
# Pré-Tramaneto
# Criação de Módulos
def dias_no_mes(data) -> int:
    '''
    Retorna o número de dias no mês da data fornecida.
    data (datetime)
    '''
    ano = data.year
    mes = data.month

    if mes == 2:
        if (ano % 4 == 0 and ano % 100 != 0) or (ano % 400 == 0):
            return 29
        else:
            return 28
    elif mes in [4, 6, 9, 11]:
        return 30
    else:
        return 31


In [ ]:
# Pré-Tratamento
# Triagem: Dados de Chuvas
linhas_preenchidas = df_chuvas[colunas_chuva].notna().any(axis=1)
linhas_vazias = ~linhas_preenchidas
print("Relação: Linhas - Dados de chuvas")
print(f"Qtd. de linhas preenchidas: {linhas_preenchidas.sum()}")
print(f"Qtd. de linhas vazias: {linhas_vazias.sum()}")

In [ ]:
# Tratamento I: Valores Ausentes em Colunas "Chuvasnn"
# Preenchimento de valores ausentes em colunas preenchidas
for i in df_chuvas[linhas_preenchidas].index:
    data = df_chuvas.loc[i, 'Data']
    num_dias = dias_no_mes(data)
    colunas_validas = [f'Chuva{dd:02d}' for dd in range(1, num_dias + 1)]
    valores = df_chuvas.loc[i, colunas_validas]
    mediana = valores.median()
    df_chuvas.loc[i, colunas_validas] = valores.fillna(mediana)

print("Preenchimento de valores ausentes nas linhas preenchidas do dataFrame temporário de Chuvas concluído.")
print("   - Linhas vazias permanecem com NaN no dataFrame.")

In [ ]:
# Tratamento II: Valores Ausentes em Colunas Maxima, Total e NumDiasDeChuva
# Criação de colunas retificadas
df_chuvas['Maxima_RET'] = df_chuvas[colunas_chuva].max(axis=1)
df_chuvas['Total_RET'] = df_chuvas[colunas_chuva].sum(axis=1)
df_chuvas['NumDiasDeChuva_RET'] = (df_chuvas[colunas_chuva] > 0).sum(axis=1)
print("Criação de colunas retificadas no dataFrame temporário de Chuvas concluida")

#### 4.3.3 Criação de dataFrame ajustado

In [ ]:
# Definição das colunas essenciais de trabalho
COLUNAS_ESSENCIAIS = ['EstacaoCodigo', 'NivelConsistencia', 'Data', 'TipoMedicaoChuvas', 'Maxima', 'Total', 'DiaMaxima', 'NumDiasDeChuva']

# Criação de dataFrame ajustado
df_ajustado = df[COLUNAS_ESSENCIAIS + colunas_chuva].copy()
print("DataFrame ajustado criado com sucesso!")

# Exibição do dataFrame ajustado
df_ajustado.head()

In [ ]:
# Atualização de colunas
df_ajustado['Maxima'] = df_chuvas['Maxima_RET']
df_ajustado['Total'] = df_chuvas['Total_RET']
df_ajustado['NumDiasDeChuva'] = df_chuvas['NumDiasDeChuva_RET']
df_ajustado[colunas_chuva] = df_chuvas[colunas_chuva]
print("Atualização das colunas do dataFrame ajustado concluidas com sucesso!")

#### 4.3.4 Tratamento de Valores Ausentes no DataFrame Ajustado

In [ ]:
# Definição de medianas
mediana_maxima = df_ajustado['Maxima'].median()
mediana_total = df_ajustado['Total'].median()
mediana_dias_chuva = df_ajustado['NumDiasDeChuva'].median()
print("Medianas definidas com sucesso")

# Retificação de valores ausentes
df_ajustado['Maxima'] = df_ajustado['Maxima'].fillna(mediana_maxima)
df_ajustado['Total'] = df_ajustado['Total'].fillna(mediana_total)
df_ajustado['NumDiasDeChuva'] = df_ajustado['NumDiasDeChuva'].fillna(mediana_dias_chuva)
print("\nValores ausentes nas colunas 'Maxima', 'Total', 'NumDiasDeChuva' preenchidos com sucesso")

In [ ]:
# Verificação final do dataFrame ajustado
print("Verificação final - Valores ausentes nas colunas principais:")
print(df_ajustado[['Maxima', 'Total', 'NumDiasDeChuva']].isnull().sum())

## **5 Biblioteca Pandas**

### 5.1 Funções `head()` e `tail()`

In [ ]:
# Função head()
print("Exibindo as primeiras linhas do dataFrame ajustado:")
df_ajustado.head()

In [ ]:
# Função tail()
print("Exibindo as cinco últimas linhas do dataFrame ajustado:")
df_ajustado.tail()

### 5.2 Função `info()`

In [ ]:
print("Exibindo informações do dataFrame ajustado:")
df_ajustado.info()

### 5.3 Parâmetro `shape`

In [ ]:
# Parâmetro shape
print("Exibindo o shape do dataFrame ajustado:")
df_ajustado.shape

### 5.4 Função `describe()`

In [ ]:
# Função describe()
print("Exibindo a estatística descritiva do dataFrame ajustado:")
print("   - Aplicada apenas as principais colunas numéricas")
df_ajustado[colunas_num_principais].describe()

### 5.5 Função `isnull()`

In [ ]:
# Função isnull()
print("Valores ausentes nas colunas principais:")
df_ajustado[colunas_num_principais].isnull().sum()

### 5.6 Funções `duplicated()` e `drop_duplicates()`

In [ ]:
# Funções duplicates() e drop_duplicates()
print("Verificação de linhas duplicadas:")
if (df_ajustado.duplicated().sum()) > 0:
    df_ajustado = df_ajustado.drop_duplicates()
    print("\nLinhas duplicadas removidas:")
    print(f"   - Qtd. de linhas: {df_ajustado.shape[0]}")
else:
    print("\nNão havia linhas duplicadas")

### 5.7 Funções `groupby()`, `mean()` e `median()`

In [ ]:
# Medidas de Tendência Central de Maxima, Total e NumDiasDeChuva por Nivel de Consistencia
var_categorica = 'NivelConsistencia'
print(f"Medidas de Tendência Central das principais\ncolunas numéricas por '{var_categorica}':")
print("------------------------------------------------")
for col in colunas_num_principais:
    print(f"Coluna: '{col}'")
    print("\nMédia:")
    print(df_ajustado.groupby(var_categorica)[col].mean())
    print("\nMediana")
    print(df_ajustado.groupby(var_categorica)[col].median())
    print("------------------------------------------------")


### 5.8 Função `value_counts()`

In [ ]:
print("Quantidade de registros por NivelConsistencia:")
df_ajustado['NivelConsistencia'].value_counts()

### 5.9 Função `sort_values()`

In [ ]:
# Função sort_values()
print("Datas com maior Total de chuvas (top 5):")
df_ajustado.sort_values('Total', ascending=False)[['Data', 'Total']].head(5)

### 5.10 Função `sort_index()`

In [ ]:
# Função sort_index()
mes = df_ajustado['Data'].dt.month
media_mes = df_ajustado.groupby(mes)['Total'].mean()
print("Média de chuvas ordenadas por mês")
media_mes.sort_index()

### 5.11 Função `corr()`

In [ ]:
# Função corr()
print("Correlação entre 'NumDiasDeChuva' e 'Total':")
correlacao = df_ajustado['NumDiasDeChuva'].corr(df_ajustado['Total'])
print(f"\nCorrelação: {correlacao:.4f}")

### 5.12 Filtros

In [ ]:
# Aplicando Filtros
valor_mm = 500
print(f"Datas com 'Total' de chuva acima de {valor_mm} mm")
print(df_ajustado[df_ajustado['Total'] > valor_mm][['Data', 'Total']])


### 5.13 Seleção de Colunas

In [ ]:
print("Selecionano apenas Data, Maxima e Total")
df_ajustado[['Data', 'Maxima', 'Total']].head()

## **6 Feature Engineering**

### 6.1 Coluna 'Estação'

In [ ]:
# Modulo
def estacao_ano(mes):
    '''
    Classifica o mês em uma das quatro estações do ano.
    Regra:
        - Dezembro, Janeiro, Fevereiro - Verão
        - Março, Abril, Maio - Outono
        - Junho, Julho, Agosto - Inverno
        - Setembro, Outubro, Novembro - Primavera
    '''
    if mes in [12, 1, 2]:
        return 'Verão'
    elif mes in [3, 4, 5]:
        return 'Outono'
    elif mes in [6, 7, 8]:
        return 'Inverno'
    else:
        return 'Primavera'
print("Função 'estacao_ano' criada com sucesso!")

In [ ]:
# Criação da coluna 'Estacao'
df_ajustado['Estacao'] = df_ajustado['Data'].dt.month.apply(estacao_ano)
print("Variável 'Estacao' criada com sucesso!")
print("   - Regra: classificação sazonal dos meses")
print("   - Utilidade: permite analisar padrões sazonais de chuva")

# Visualizar coluna
df_ajustado['Estacao'].head()